In [14]:
import os
import yaml

# Dataset Fetching

In [15]:
yaml_path = r"D:\Himanshu_ML\TE_UAV Data\TE_Bdy v3\dataset.yaml"

with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

print("Classes:", data["names"], "nc:", {len(data["names"])})
print("Train images path:", data["train"])
print("Validation images path:", data["val"])


Classes: ['UAV'] nc: {1}
Train images path: images/train
Validation images path: images/valid


#  Data Preprocessing

In [16]:
import cv2
import glob
from tqdm import tqdm

Finding Corrupted Dataset

In [ ]:
img_dir=data["train"]

all_imgs=glob.glob(os.path.join(img_dir,"*.jpg", "*.jpeg", "*.png"))

bad_imgs=[]

for img_pth in tqdm(all_imgs):
    try:
        img=cv2.imread(img_pth)
        if img is None:
            bad_imgs.append(img_pth)
    except:
        bad_imgs.append(img_pth)
print("Corrupt:", bad_imgs)



0it [00:00, ?it/s]

Corrupt: []


# Data Augmentation

In [18]:
import albumentations  as A
from albumentations.pytorch import ToTensorV2
import numpy as np

In [19]:
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.3),
    A.RandomFog(p=0.2),
    A.RandomRain(p=0.2),
    A.GaussNoise(p=0.3),
], 
    bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"])
)


In [20]:
in_imgdir=r"D:\Himanshu_ML\TE_UAV Data\TE_Bdy v3\images\train"
in_labeldir=r"D:\Himanshu_ML\TE_UAV Data\TE_Bdy v3\labels\train"

out_img_dir = r"D:\Himanshu_ML\TE_UAV Data\TE_Bdy v3\images\train_aug"
out_lbl_dir = r"D:\Himanshu_ML\TE_UAV Data\TE_Bdy v3\labels\train_aug"

os.makedirs(out_img_dir, exist_ok=True)
os.makedirs(out_lbl_dir, exist_ok=True)


img_paths = glob.glob(os.path.join(img_dir, "*.jpg"))

for img_path in img_paths:
    base = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(in_labeldir, base + ".txt")

    if not os.path.exists(lbl_path):
        print(f"⚠️ Label missing for {img_path}, skipping.")
        continue

    # Load image + labels
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    bboxes, class_labels = [], []

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            cls = int(parts[0])
            coords = list(map(float, parts[1:]))  # [x_center, y_center, width, height] in YOLO
            bboxes.append(coords)
            class_labels.append(cls)

    # Apply augmentation
    augmented = transform(image=img, bboxes=bboxes, class_labels=class_labels)
    aug_img = augmented["image"]
    aug_bboxes = augmented["bboxes"]
    aug_labels = augmented["class_labels"]

    # Save augmented image
    new_img_path = os.path.join(out_img_dir, f"{base}_aug.jpg")
    cv2.imwrite(new_img_path, aug_img)

    # Save augmented labels
    new_lbl_path = os.path.join(out_lbl_dir, f"{base}_aug.txt")
    with open(new_lbl_path, "w") as f:
        for cls, box in zip(aug_labels, aug_bboxes):
            f.write(f"{cls} " + " ".join(map(str, box)) + "\n")

print("Augmented dataset with labels created.")


Augmented dataset with labels created.


# Model Training

In [21]:
from ultralytics import YOLO
import torch, os, sys
from ultralytics.data.augment import Albumentations


In [ ]:
print("CUDA:", torch.version.cuda, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: 12.1 | CUDA available: True
GPU: NVIDIA GeForce RTX 2080 Ti


In [23]:
albumentations_aug = Albumentations(transform)

albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


In [ ]:
train_Args=dict(
    data=yaml_path,      
    name="Aug_bdy",       
    device=0,              
    epochs=120,         
    imgsz=1024,            # high res (good for UAV body)
    batch=8,               
    workers=0,         
    seed=42,
    deterministic=True, 
    amp=True,            

    optimizer="AdamW",    
    lr0=0.001,             # standard start LR
    cos_lr=True,           # cosine annealing for smoother convergence
    warmup_epochs=5,       # warmup to stabilize
    weight_decay=0.0005,   

    mosaic=0.5,
    degrees=15,
    perspective=0.001,
    translate=0.1,
    scale=0.2,
    flipud=0.1,
    fliplr=0.5,
    augment=True,
    
    val=True,            
    patience=100,          
    plots=True,            # saves PR/mAP/loss plots
    overlap_mask=True  

)


In [25]:
# just load weights
model = YOLO(r"D:\Himanshu_ML\TE_UAV Projects\TE_1 UAV MRO\YOLO_Model\yolo11l-seg.pt")

In [26]:
results=model.train(**train_Args)

Ultralytics 8.3.192  Python-3.11.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 11264MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\Himanshu_ML\TE_UAV Data\TE_Bdy v3\dataset.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\Himanshu_ML\TE_UAV Projects\TE_1 UAV MRO\YOLO_Model\yolo11l-seg.pt, momentum=0.937, mosaic=0.5, multi_scale=False, name=Aug_bdy3, nbs=64, nms=False, opset=None, optimize=False, opt

# Model Evaluation

In [1]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

In [2]:
model = YOLO(r"runs/segment/Aug_bdy3/weights/best.pt")

In [3]:
metrics=model.val(data="D:\Himanshu_ML\TE_UAV Data\TE_Bdy v3\dataset.yaml", imgsz=1024)
print(metrics)


Ultralytics 8.3.192  Python-3.11.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2080 Ti, 11264MiB)
YOLO11l-seg summary (fused): 203 layers, 27,585,363 parameters, 0 gradients, 131.8 GFLOPs
val: Fast image access  (ping: 0.30.0 ms, read: 271.97.8 MB/s, size: 3630.8 KB)
val: Scanning D:\Himanshu_ML\TE_UAV Data\TE_Bdy v3\labels\valid.cache... 75 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 75/75  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 0.53it/s 9.5s1.1s
                   all         75         75      0.967          1      0.994      0.891      0.967          1      0.994       0.87
Speed: 3.8ms preprocess, 55.9ms inference, 0.0ms loss, 5.2ms postprocess per image
Results saved to runs\segment\val2
ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralyt

Visualization

%matplotlib qt


%matplotlib inline   # shows plots inside VS Code notebook

In [7]:
precision = float(metrics.box.p.mean())
recall = float(metrics.box.r.mean())
map50 = float(metrics.box.map50)
map5095 = float(metrics.box.map)

seg_map50 = float(metrics.seg.map50)
seg_map5095 = float(metrics.seg.map)

labels = ["Precision", "Recall", "mAP50 (Box)", "mAP50-95 (Box)", "mAP50 (Seg)", "mAP50-95 (Seg)"]
values = [precision, recall, map50, map5095, seg_map50, seg_map5095]

In [15]:
plt.figure(figsize=(10,6))
bars = plt.bar(labels, values, color=["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b"])
plt.ylim(0,1)
plt.ylabel("Score")
plt.title("YOLO Evaluation Metrics")

# Annotate values on top of bars
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}", ha="center")

plt.show()

C:\Users\TE\AppData\Local\Temp\ipykernel_2916\1397239174.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Model Testing

In [1]:
from ultralytics import YOLO

In [2]:
model = YOLO(r"runs/segment/Aug_bdy3/weights/best.pt")

In [3]:
results = model.predict(r"C:\Users\TE\Downloads\IMG_20250910_182205938 (1).jpg", imgsz=1024, conf=0.2, save=True)


image 1/1 C:\Users\TE\Downloads\IMG_20250910_182205938 (1).jpg: 1024x800 2 UAVs, 42.6ms
Speed: 6.8ms preprocess, 42.6ms inference, 70.2ms postprocess per image at shape (1, 3, 1024, 800)
Results saved to runs\segment\predict8
